In [ ]:
import requests
import pandas as pd
import re

# ------------------ CONFIG ------------------
TBA_API_KEY = "ENTER API KEY HERE"
HEADERS = {"X-TBA-Auth-Key": TBA_API_KEY}

YEAR = 2025
# --------------------------------------------


def team_key_to_int(team_key):
    """frc254 -> 254, frc4788B -> 4788"""
    match = re.search(r"\d+", team_key)
    return int(match.group()) if match else None


rows = []

# Get all events for the year
events_url = f"https://www.thebluealliance.com/api/v3/events/{YEAR}"
events = requests.get(events_url, headers=HEADERS).json()

print(f"Found {len(events)} events in {YEAR}")

for i, event in enumerate(events, start=1):
    event_key = event["key"]
    event_name = event["name"]

    print(f"Processing {i}/{len(events)}: {event_key}")

    matches_url = f"https://www.thebluealliance.com/api/v3/event/{event_key}/matches"
    matches_resp = requests.get(matches_url, headers=HEADERS)

    if matches_resp.status_code != 200:
        continue

    matches = matches_resp.json()

    for match in matches:
        alliances = match.get("alliances")
        score_breakdown = match.get("score_breakdown")
        videos = match.get("videos", [])

        if not alliances or not score_breakdown or not videos:
            continue

        red = alliances["red"]
        blue = alliances["blue"]

        red_teams = [team_key_to_int(t) for t in red["team_keys"]]
        blue_teams = [team_key_to_int(t) for t in blue["team_keys"]]

        red_score = red.get("score")
        blue_score = blue.get("score")

        # Autonomous points (field name is season-specific but usually autoPoints)
        red_auto = score_breakdown["red"].get("autoPoints")
        blue_auto = score_breakdown["blue"].get("autoPoints")

        for video in videos:
            if video.get("type") == "youtube":
                rows.append({
                    "youtube_link": f"https://www.youtube.com/watch?v={video['key']}",
                    "event_name": event_name,
                    
                    "red_team_1": red_teams[0],
                    "red_team_2": red_teams[1],
                    "red_team_3": red_teams[2],
                    "blue_team_1": blue_teams[0],
                    "blue_team_2": blue_teams[1],
                    "blue_team_3": blue_teams[2],

                    "red_auto_points": red_auto,
                    "blue_auto_points": blue_auto,

                    "red_score": red_score,
                    "blue_score": blue_score,
                })

df = pd.DataFrame(rows)

print(f"Total matches collected: {len(df)}")
print(df.head())

df.to_csv("frc_2025_matches.csv", index=False)


: 